In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType,
    DoubleType, IntegerType, ArrayType
)
from pyspark.sql.functions import current_timestamp, lit

# 1. Define Strict Schema (Schema-on-Read)

hourly_units_schema = StructType([
    StructField("time", StringType(), True),
    StructField("carbon_monoxide", StringType(), True),
    StructField("carbon_dioxide", StringType(), True),
    StructField("pm2_5", StringType(), True),
    StructField("pm10", StringType(), True),
    StructField("nitrogen_dioxide", StringType(), True)
])

hourly_schema = StructType([
    StructField("time", ArrayType(StringType()), True),
    StructField("carbon_monoxide", ArrayType(DoubleType()), True),
    StructField("carbon_dioxide", ArrayType(DoubleType()), True),
    StructField("pm2_5", ArrayType(DoubleType()), True),
    StructField("pm10", ArrayType(DoubleType()), True),
    StructField("nitrogen_dioxide", ArrayType(DoubleType()), True)
])

open_meteo_schema = StructType([
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("generationtime_ms", DoubleType(), True),
    StructField("utc_offset_seconds", IntegerType(), True),
    StructField("timezone", StringType(), True),
    StructField("timezone_abbreviation", StringType(), True),
    StructField("elevation", DoubleType(), True),
    StructField("hourly_units", hourly_units_schema, True),
    StructField("hourly", hourly_schema, True),
    StructField("city", StringType(), True),
    StructField("source_system", StringType(), True)
])

print("Strict schema defined successfully and matches the verified JSON payload.")

In [0]:
CATALOG_VOLUME_ROOT = "/Volumes/workspace/default"

bronze_delta_path = f"{CATALOG_VOLUME_ROOT}/staging/bronze_carbon_data"
log_table_path    = f"{CATALOG_VOLUME_ROOT}/staging/pipeline_execution_logs"
raw_folder_path   = f"{CATALOG_VOLUME_ROOT}/staging"   # where dated raw JSON files land

print("bronze_delta_path:", bronze_delta_path)
print("log_table_path   :", log_table_path)
print("raw_folder_path  :", raw_folder_path)

#dbutils.fs.mkdirs(raw_folder_path)
#dbutils.fs.mkdirs(f"{CATALOG_VOLUME_ROOT}/staging/raw")
#dbutils.fs.mkdirs(f"{CATALOG_VOLUME_ROOT}/staging/processed")

# Ensure the new 'staging' volume exists in Unity Catalog
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.staging")
# Sanity Check: Test the schema against a real sample file

sample_file_path = f"{raw_folder_path}/incremental_load_sample.json"

try:
    test_df = (spark.read
               .schema(open_meteo_schema)
               .option("multiline", "true")
               .json(sample_file_path))

    print("File read successfully! Here is a preview:")
    display(test_df)

except Exception as e:
    print("Error reading the file. Check your file path!")
    print(e)

In [0]:
# Create Audit Log Table (run once; safe to re-run)
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

log_schema = StructType([
    StructField("layer_processed", StringType(), True),
    StructField("parameter_processed", StringType(), True),
    StructField("start_time", TimestampType(), True),
    StructField("end_time", TimestampType(), True),
    StructField("status", StringType(), True),
    StructField("rows_processed", IntegerType(), True)
])

try:
    spark.read.format("delta").load(log_table_path)
    print("Log table already exists.")
except Exception:
    empty_df = spark.createDataFrame([], log_schema)
    empty_df.write.format("delta").save(log_table_path)
    print("Audit log table created successfully.")


# DESIGN DECISION: Bronze writes with mode="append".
# Re-running this function on the same file will duplicate raw rows in
# Bronze -- that is intentional. Bronze is an append-only audit log of
# exactly what the API returned on every call, including overlapping
# re-fetches of recent days (see TA feedback point 1 below). Idempotency
# -- i.e. "no duplicate rows downstream" -- is guaranteed at the Silver
# layer via MERGE INTO on (city, reading_time), not here.

from pyspark.sql.functions import current_timestamp, lit
from datetime import datetime
from pyspark.sql import functions as F

def ingest_raw_to_bronze(input_file_path, bronze_table_path, log_table_path):
    layer_name = "Raw-to-Bronze"
    start_time = datetime.now()
    status = "Failed"
    row_count = 0

    print(f"--- Starting {layer_name}: {input_file_path} ---")

    try:
        # 1. Read & add metadata
        raw_df = (spark.read
                  .schema(open_meteo_schema)
                  .option("multiline", "true")
                  .json(input_file_path))

        bronze_df = (raw_df
                     .withColumn("load_timestamp", current_timestamp())
                     .withColumn("source_file", F.col("_metadata.file_path")))

        # FIX #2: count captured before the write triggers a second full pass
        row_count = bronze_df.count()

        # 2. Write to Bronze (append-only -- see design decision above)
        (bronze_df.write
         .format("delta")
         .mode("append")
         .option("mergeSchema", "true")
         .save(bronze_table_path))

        status = "Success"
        print(f"Success! {row_count} rows appended to Bronze layer.")

    except Exception as e:
        print(f"Pipeline Failed: {e}")
        # FIX #3: full error message retained (was str(e)[:50])
        status = f"Failed: {str(e)[:400]}"

    finally:
        end_time = datetime.now()

        log_entry = spark.createDataFrame([(
            layer_name,
            input_file_path,
            start_time,
            end_time,
            status,
            row_count
        )], schema=log_schema)

        log_entry.write.format("delta").mode("append").save(log_table_path)
        print("Audit log updated.")
        print("---------------------------------")

    return status

In [0]:
# Parameterized Backfill Wrapper
#
# ingest_raw_to_bronze() already takes a file path as a parameter rather
# than a hardcoded one. This wrapper extends that to a DATE RANGE, so the
# same code path handles both:
#   - a single incremental day  -> backfill_bronze(today, today, ...)
#   - a full historical backfill -> backfill_bronze("2023-09-01", today, ...)
# Expects raw files named like  raw_folder/YYYY-MM-DD.json

from datetime import date, timedelta

def backfill_bronze(start_date: str, end_date: str, raw_folder: str,
                     bronze_table_path: str, log_table_path: str):
    current = date.fromisoformat(start_date)
    end = date.fromisoformat(end_date)
    results = []
    while current <= end:
        file_path = f"{raw_folder}/{current.isoformat()}.json"
        status = ingest_raw_to_bronze(file_path, bronze_table_path, log_table_path)
        results.append((current.isoformat(), status))
        current += timedelta(days=1)
    return results

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.bronze_layer")
# --- Single-file run
#source_data_parameter = f"{raw_folder_path}/incremental_load_sample.json"
#ingest_raw_to_bronze(source_data_parameter, bronze_delta_path, log_table_path)


# --- Full Load 3 years
source_data_parameter = f"{raw_folder_path}/full_load_last_3_years.json"
ingest_raw_to_bronze(source_data_parameter, bronze_delta_path, log_table_path)
display(spark.read.format("delta").load(log_table_path))